In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/telco_cleaned.csv')


def tenure_bucket(months):
    if months <= 12:
        return 'new'
    elif months <= 36:
        return 'mid'
    else:
        return 'long'

df['tenure_group'] = df['tenure'].apply(tenure_bucket)

X = df.drop('Churn', axis=1)
y = df['Churn'].map({'Yes': 1, 'No': 0})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (5634, 20)
Test shape: (1409, 20)


In [3]:
categorical_cols = X.select_dtypes(include='object').columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)

Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'tenure_group']
Numerical columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']


C:\Users\ACER\AppData\Local\Temp\ipykernel_17916\2785446876.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include='object').columns.tolist()


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols)
    ]
)

In [5]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Train processed shape:", X_train_processed.shape)
print("Test processed shape:", X_test_processed.shape)

Train processed shape: (5634, 32)
Test processed shape: (1409, 32)


In [6]:
import joblib

joblib.dump(preprocessor, 'preprocessor.pkl')

X_train.to_csv('../data/X_train_raw.csv', index=False)
X_test.to_csv('../data/X_test_raw.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

import numpy as np
np.save('../data/X_train_processed.npy', X_train_processed)
np.save('../data/X_test_processed.npy', X_test_processed)

In [7]:
feature_names = (
    numerical_cols + 
    list(preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols))
)

print("Total processed features:", len(feature_names))
print(feature_names[:10])  

import json
with open('../data/feature_names.json', 'w') as f:
    json.dump(feature_names, f)

Total processed features: 32
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_No phone service', 'MultipleLines_Yes']


In [8]:
print(preprocessor)

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['SeniorCitizen', 'tenure', 'MonthlyCharges',
                                  'TotalCharges']),
                                ('cat',
                                 OneHotEncoder(drop='first',
                                               handle_unknown='ignore'),
                                 ['gender', 'Partner', 'Dependents',
                                  'PhoneService', 'MultipleLines',
                                  'InternetService', 'OnlineSecurity',
                                  'OnlineBackup', 'DeviceProtection',
                                  'TechSupport', 'StreamingTV',
                                  'StreamingMovies', 'Contract',
                                  'PaperlessBilling', 'PaymentMethod',
                                  'tenure_group'])])


In [9]:
import os
import joblib

os.makedirs('../models', exist_ok=True)
joblib.dump(preprocessor, '../models/preprocessor.pkl')

print(os.listdir('../models'))

['logistic_regression.pkl', 'model_config.json', 'preprocessor.pkl', 'random_forest.pkl', 'shap_explainer.pkl', 'xgboost_tuned.pkl']
